# rank-world-size-args — ex1: thread rank and world_size through a broadcast signature

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rank-world-size-args`. Running the final beacon cell reports progress against the `Distributed: rank/world_size args` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: rank/world_size args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rank-world-size-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rank-world-size-args"
DD_SUBTOPIC = "Distributed: rank/world_size args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.distributed quick refresher

PyTorch's collective-communication library (`torch.distributed`, aliased `dist`) lets multiple processes coordinate over tensors. The standard workflow:

1. **Each rank** runs the same function, parameterized by `rank` and `world_size`. Rank 0 is conventionally the 'driver'.
2. **`dist.init_process_group(backend=...)`** establishes the rendezvous. Backends:
   - `'nccl'` — NVIDIA's GPU-to-GPU primitive. Used in ARENA's multi-GPU setup. Requires CUDA + one process per GPU.
   - `'gloo'` — CPU-friendly. What you'll use in these drills (Colab CPU runtimes have no real GPUs).
3. **Pin a device** per rank: `torch.device(f'cuda:{rank}')` so each process owns exactly one GPU.
4. **Collective ops** (`all_reduce`, `broadcast`, `send`, `recv`) operate in-place on tensors of identical shape across all ranks.
5. **`dist.destroy_process_group()`** tears down at the end.

**Two ways to launch multiple ranks:**
- `torch.multiprocessing.spawn(fn, args=(...), nprocs=world_size)` — what ARENA uses. Spawn requires the worker fn be importable (not defined in `__main__`/a notebook cell).
- `mp.get_context('fork').Process(target=fn, args=...)` — Linux-only but works with cell-defined fns. The drills use this in tests so the worker can stay in the cell.

**Two-rank trick.** Colab gives ~2 CPU cores, so `world_size=2` is the right scale: enough to exercise the protocol, cheap enough to finish in seconds.

### This drill's atom: explicit `(rank, world_size)` plumbing
Every distributed-aware function in ARENA's chap-0 part-3 has `rank` and `world_size` in its signature — `broadcast(tensor, rank, world_size, src=0)`, `reduce(tensor, rank, world_size, dst, op)`, etc. **Why repeat them everywhere?**
- The launcher (`mp.spawn`) passes `rank` to the worker as the first positional arg. You then thread it through every call so each function can decide what THIS rank does.
- `world_size` is constant across all ranks but needed for loops like `for other_rank in range(world_size):`.
- Functions could read them from `dist.get_rank()` / `dist.get_world_size()`, but explicit args are cheaper, test-friendlier (you can call them without an active process group), and make the data-flow obvious.

### Exercise 1 — thread rank and world_size through a broadcast signature

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `(tensor, rank, world_size, src=0)` signature convention from ARENA's broadcast — branch on `rank == src` to decide between sending and receiving, looping `range(world_size)` for the fanout.
> Keywords: rank, world_size, signature, src, broadcast-protocol
> ```

**KCs targeted:** `rank-world-size-positional-args`, `src-rank-branching`

Implement `ex1_broadcast_protocol(tensor, rank, world_size, src=0)` — a **pure-logic** stand-in for `dist.broadcast` that does NOT actually transmit. Instead, it returns a list of `(action, other_rank)` tuples describing what THIS rank would do, so we can test the protocol without a real process group.

Spec:
- If `rank == src`: for each `other_rank in range(world_size)` where `other_rank != src`, append `('send', other_rank)`. Order matters — must be ascending by `other_rank`.
- If `rank != src`: return `[('recv', src)]`.
- `tensor` is unused by this dry-run protocol — it's only there to mirror the real signature.

Return the list of `(action, other_rank)` tuples.

This drill is the **signature** atom — once you've internalized `(tensor, rank, world_size, src=0)`, the matching `reduce(tensor, rank, world_size, dst=0, op)` and `all_reduce(tensor, rank, world_size, op)` slot into muscle memory.

In [ ]:
import torch as t
from torch import Tensor

def ex1_broadcast_protocol(tensor: Tensor, rank: int, world_size: int, src: int = 0):
    """Pure-logic broadcast protocol. Returns list[(action, other_rank)]."""
    raise NotImplementedError()


def _test_ex1():
    dummy = t.tensor([1.0])

    # rank 0 = src, world_size=3 → sends to 1 and 2 in order.
    assert ex1_broadcast_protocol(dummy, rank=0, world_size=3, src=0) == [
        ('send', 1), ('send', 2)
    ]
    # rank 1 (not src) → recv from src=0.
    assert ex1_broadcast_protocol(dummy, rank=1, world_size=3, src=0) == [('recv', 0)]
    assert ex1_broadcast_protocol(dummy, rank=2, world_size=3, src=0) == [('recv', 0)]

    # Custom src=2 → rank 2 sends to 0, 1, 3; others recv from 2.
    assert ex1_broadcast_protocol(dummy, rank=2, world_size=4, src=2) == [
        ('send', 0), ('send', 1), ('send', 3)
    ]
    for non_src in [0, 1, 3]:
        assert ex1_broadcast_protocol(dummy, rank=non_src, world_size=4, src=2) == [('recv', 2)]

    # Single-rank degenerate world_size=1 → src is the only rank, no sends.
    assert ex1_broadcast_protocol(dummy, rank=0, world_size=1, src=0) == []

    # Signature check: src defaults to 0.
    import inspect
    sig = inspect.signature(ex1_broadcast_protocol)
    params = list(sig.parameters.values())
    names = [p.name for p in params]
    assert names == ['tensor', 'rank', 'world_size', 'src'], f'signature order wrong: {names}'
    assert sig.parameters['src'].default == 0, 'src must default to 0'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_broadcast_protocol(tensor: Tensor, rank: int, world_size: int, src: int = 0):
    if rank == src:
        return [('send', other) for other in range(world_size) if other != src]
    return [('recv', src)]
```

**The src/dst convention.** `src` for ops where one rank pushes to many (`broadcast`, `scatter`). `dst` for ops where many ranks push to one (`reduce`, `gather`). `all_*` variants drop both because every rank is both source and destination.

**Why dry-run-able protocols are a debugging superpower.** When your real `broadcast` hangs in production, you cannot inspect what each rank was trying to do (the process is stuck inside C++). Writing a parallel pure-logic version like this one lets you call it for every `(rank, world_size, src)` combo and confirm the intended message pattern. ARENA does exactly this when demonstrating the broadcast diagram in the markdown.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()